#The Transformation Logic

In [0]:
catalog = "`abhi-de-ete-project-catlog`"


In [0]:
spark.sql(f"create schema if not exists {catalog}.gold")

In [0]:
spark.sql(f"select * from {catalog}.silver.crm_customers limit 5").display()
spark.sql(f"select * from {catalog}.silver.erp_customers limit 5").display()
spark.sql(f"select * from {catalog}.silver.erp_customer_location limit 5").display()

In [0]:
query = f"""

select 
  row_number() over (order by ci.customer_id ) as customer_key,
  ci.customer_id,
  ci.customer_number,
  ci.first_name,
  ci.last_name,
  la.country,
  ci.marital_status,
  case 
    when ci.gender != 'n/a' then ci.gender
    else coalesce (ca.gender, 'n/a')
  end as gender,
  ca.birth_date as birthdate,
  ci.created_date as create_date 
from  {catalog}.silver.crm_customers ci
left join  {catalog}.silver.erp_customers ca
  on ci.customer_number = ca.customer_number
left join  {catalog}.silver.erp_customer_location la
  on ci.customer_number  = la.customer_number

"""

In [0]:
df = spark.sql(query)

In [0]:
df.limit(10).display()

#Writing Gold Table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable(f"{catalog}.gold.dim_customers")

## Sanity checks of Gold table

In [0]:
%sql
SELECT * FROM `abhi-de-ete-project-catlog`.gold.dim_customers LIMIT 10